# EJERCICIO DE ANALISIS DE SENTIMIENTOS USANDO EMBEDDINGS PREENTRENADOS Y REDES NEURONALES CON TENSORFLOW

# instalamos gensim

In [1]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 50.8 MB/s eta 0:00:00


# importamos librerias

In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, GlobalAveragePooling1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from gensim.models import Word2Vec

# creamos dataset de prueba

In [3]:
# 🔹 Simulación de carga de un dataset (puedes reemplazar por un CSV real)
tweets = [
    'Este producto es una maravilla',
    'No recomiendo comprar esto',
    'Excelente atención y calidad',
    'Muy mala experiencia, pésimo servicio',
    'Totalmente satisfecho con mi compra',
    'No funciona como esperaba',
    'Una compra perfecta y rápida',
    'Decepcionante, esperaba algo mejor'
]

# 1 = positivo, 0 = negativo
labels = [1, 0, 1, 0, 1, 0, 1, 0]

# TOKENIZACION

In [4]:
# 🔹 Tokenización
tokenizer = Tokenizer()
# Fit tokenizer on lowercased tweets to align with Word2Vec vocabulary
tweets_tokenized = [tweet.lower().split() for tweet in tweets]
tokenizer.fit_on_texts([' '.join(tweet) for tweet in tweets_tokenized])
sequences = tokenizer.texts_to_sequences([' '.join(tweet) for tweet in tweets_tokenized])
word_index = tokenizer.word_index
word_index

{'una': 1,
 'no': 2,
 'y': 3,
 'compra': 4,
 'esperaba': 5,
 'este': 6,
 'producto': 7,
 'es': 8,
 'maravilla': 9,
 'recomiendo': 10,
 'comprar': 11,
 'esto': 12,
 'excelente': 13,
 'atención': 14,
 'calidad': 15,
 'muy': 16,
 'mala': 17,
 'experiencia': 18,
 'pésimo': 19,
 'servicio': 20,
 'totalmente': 21,
 'satisfecho': 22,
 'con': 23,
 'mi': 24,
 'funciona': 25,
 'como': 26,
 'perfecta': 27,
 'rápida': 28,
 'decepcionante': 29,
 'algo': 30,
 'mejor': 31}

In [5]:
padded_sequences = pad_sequences(sequences, padding='post')

# ENTRENAR LOS TWEETS TOKENIZADOS CON WORD2VEC

In [6]:
w2v_model = Word2Vec(tweets_tokenized, vector_size=100, window=3, min_count=1, sg=1)

# CREO UNA MATRIZ DE EMBEDDING

In [7]:
vocab_size = len(word_index) + 1
embedding_dim = 100
embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, i in word_index.items():
    # Check if the word is in the Word2Vec model's vocabulary before accessing
    if word in w2v_model.wv:
        embedding_vector = w2v_model.wv[word]
        embedding_matrix[i] = embedding_vector

In [8]:
embedding_matrix

array([[ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [-0.00714151,  0.00124363, -0.00717742, ...,  0.00481513,
         0.00078795,  0.00301512],
       [-0.00824268,  0.00929935, -0.00019766, ..., -0.00744759,
        -0.00250607, -0.00554986],
       ...,
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.00813019, -0.00444857, -0.00106611, ..., -0.00574926,
        -0.00166619,  0.00558429],
       [-0.00870784,  0.00211635, -0.0008619 , ..., -0.00870141,
         0.0029575 , -0.0066712 ]])

# CREAMOS EL MODELO DE RED NEURONAL CON TENSORFLOW

In [9]:
model = Sequential(
    [
        Embedding(
            input_dim=vocab_size,
            output_dim=embedding_dim,
            input_length=padded_sequences.shape[1],
            weights=[embedding_matrix],
            trainable=False),
        GlobalAveragePooling1D(),
        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid')
    ]
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


# COMPILAMOS EL MODELO

In [10]:
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

# ENTRENAMOS EL MODELO

In [11]:
model.fit(padded_sequences, np.array(labels), epochs=30)

Epoch 1/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.5000 - loss: 0.6929
Epoch 2/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.6250 - loss: 0.6922
Epoch 3/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.6250 - loss: 0.6915
Epoch 4/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.6250 - loss: 0.6910
Epoch 5/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.8750 - loss: 0.6905
Epoch 6/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.8750 - loss: 0.6901
Epoch 7/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.8750 - loss: 0.6898
Epoch 8/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.8750 - loss: 0.6895
Epoch 9/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.8750 - loss: 0.6893
Epoch 10/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.8750 - loss: 0.6891
Epoch 11/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 1.0000 - loss: 0.6889
Epoch 12/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 1.0000 - loss: 0.6886
Epo

# PROBAMOS EL MODELO

In [12]:
# 🔹 Predicción sobre un tweet nuevo
tweet_nuevo = ['pésimo producto, muy decepcionado']
seq_nuevo = tokenizer.texts_to_sequences(tweet_nuevo)
seq_nuevo_padded = pad_sequences(seq_nuevo, maxlen=padded_sequences.shape[1], padding='post')
prediccion = model.predict(seq_nuevo_padded)

print('Probabilidad de ser positivo:', float(prediccion[0]))
print('Sentimiento:', 'Positivo' if prediccion[0] > 0.5 else 'Negativo')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
Probabilidad de ser positivo: 0.49946919083595276
Sentimiento: Negativo


/tmp/ipykernel_2182/1103175599.py:7: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  print('Probabilidad de ser positivo:', float(prediccion[0]))
